# Legal NLP in the LLM Era

**NLPAICS 2026 Summer School — The Paradigm Shift** · Day 3 · Wednesday 17 June 2026

**Lecturer:** Damith Premasiri

> Before running anything, make sure the kernel is **NLPAICS 10** (menu: *Kernel → Change Kernel*). It should already be selected.

## 0 · Environment check

In [ ]:
# --- Environment check: run this cell first ---------------------------------
# It verifies you are on this lesson's kernel and that the GPU is visible.
import sys

assert ".venv" in sys.executable, (
    "Wrong kernel! In the menu choose: Kernel > Change Kernel > 'NLPAICS 10"
)
print("Kernel OK:", sys.executable)

try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed (fine if this lesson doesn't need it)")


---

## 1 · Legal Retrieval — Why it Matters

Modern legal AI systems are built on **retrieval-augmented generation (RAG)**:
a user asks a legal question → relevant contract clauses / case passages are retrieved
→ those passages are fed to an LLM → a grounded answer is produced.

This practical covers the **retrieval** half of that pipeline:

| Stage | Method | Library |
|---|---|---|
| Sparse retrieval | BM25 | `rank_bm25` |
| Dense retrieval (general) | Sentence-BERT (MiniLM) | `sentence-transformers` |
| Dense retrieval (legal) | Legal-BERT (mean pool) | `transformers` |
| Index & search | FAISS | `faiss-cpu` |

**Dataset:** [CUAD](https://huggingface.co/datasets/theatticusproject/cuad-qa) — 500 commercial contracts
with 13 000+ expert-labelled clause annotations.  We use a subset that fits comfortably in GPU memory
and runs end-to-end in ≈ 90 minutes.

## 2 · Imports

In [ ]:
import json, re, textwrap, warnings
from collections import defaultdict

import numpy as np
import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU    : {torch.cuda.get_device_name(0)}  ({gb:.1f} GB)")

## 3 · The CUAD Dataset

CUAD (Contract Understanding Atticus Dataset) pairs **contract passages** with
**legal questions** and **ground-truth answer spans**.

We will use it as a retrieval benchmark:
- **Corpus** = all unique contract passages (chunked)
- **Queries** = the legal questions
- **Relevance** = passages that contain the annotated answer span

In [ ]:
from datasets import load_dataset

# Load only the test split (~4 500 Q/A pairs) to keep runtime short
raw = load_dataset("theatticusproject/cuad-qa", split="test", trust_remote_code=True)
print(f"Total QA pairs : {len(raw)}")
print("\nColumn names   :", raw.column_names)

In [ ]:
# ── Inspect one example ───────────────────────────────────────────────────────
ex = raw[0]
print("Contract title :", ex["title"])
print("Question       :", ex["question"])
print("Answer text    :", ex["answers"]["text"])
print("\nContext snippet (first 400 chars):")
print(ex["context"][:400])

In [ ]:
# ── How many unique contracts are there? ─────────────────────────────────────
titles = list({ex["title"] for ex in raw})
print(f"Unique contracts in test split : {len(titles)}")

# Keep the first N contracts to stay within our time budget
N_CONTRACTS = 80   # ← increase to 200+ if you have more time
kept_titles = set(titles[:N_CONTRACTS])
subset = raw.filter(lambda x: x["title"] in kept_titles)
print(f"QA pairs after filtering       : {len(subset)}")
print(f"Unique contracts kept          : {len({x['title'] for x in subset})}")

## 4 · Building the Passage Corpus

BERT-family models accept at most **512 tokens**. We split each contract into
overlapping *passages* (chunks) so that answer spans are unlikely to straddle a
boundary.

```
chunk_size    = 256 words
chunk_overlap =  64 words
```

In [ ]:
def chunk_text(text: str, chunk_size: int = 256, overlap: int = 64) -> list:
    """Split text into overlapping word-level windows."""
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start += chunk_size - overlap
    return chunks


# Build corpus: lists of passage text and metadata
corpus_texts = []       # passage text
corpus_meta  = []       # (contract_title, chunk_index)
contract_to_chunks = defaultdict(list)   # title -> [passage_indices]

seen = set()
for ex in subset:
    title = ex["title"]
    if title in seen:
        continue
    seen.add(title)
    for i, ch in enumerate(chunk_text(ex["context"])):
        idx = len(corpus_texts)
        corpus_texts.append(ch)
        corpus_meta.append((title, i))
        contract_to_chunks[title].append(idx)

print(f"Total passages in corpus : {len(corpus_texts)}")
print(f"Avg words per passage    : {np.mean([len(t.split()) for t in corpus_texts]):.0f}")

In [ ]:
# ── Build evaluation ground truth ─────────────────────────────────────────────
# A passage is relevant if the gold answer span appears in it.
queries, qrel = [], []

for ex in subset:
    answer_texts = ex["answers"]["text"]
    if not answer_texts:
        continue                     # unanswerable query — skip
    relevant_idxs = []
    for pid in contract_to_chunks[ex["title"]]:
        if any(ans.strip() and ans.strip() in corpus_texts[pid]
               for ans in answer_texts):
            relevant_idxs.append(pid)
    if not relevant_idxs:
        continue                     # answer not found in any chunk — skip
    queries.append(ex["question"])
    qrel.append(relevant_idxs)

print(f"Evaluable queries         : {len(queries)}")
print(f"Avg relevant passages / q : {np.mean([len(r) for r in qrel]):.2f}")
print(f"\nExample query : {queries[0]}")
print(f"Relevant IDs  : {qrel[0]}")

## 5 · Evaluation Metrics

| Metric | Meaning |
|---|---|
| **Recall@k** | fraction of queries with ≥ 1 relevant passage in top-k |
| **MRR@k** | mean of 1/rank of first relevant passage |

Both are standard in the BEIR retrieval benchmark.

In [ ]:
def recall_at_k(ranked_lists, qrels, k):
    hits = sum(
        any(pid in rel for pid in ranked[:k])
        for ranked, rel in zip(ranked_lists, qrels)
    )
    return hits / len(ranked_lists)


def mrr_at_k(ranked_lists, qrels, k):
    rr = 0.0
    for ranked, rel in zip(ranked_lists, qrels):
        for rank, pid in enumerate(ranked[:k], 1):
            if pid in rel:
                rr += 1.0 / rank
                break
    return rr / len(ranked_lists)


def evaluate(ranked_lists, qrels, ks=(1, 5, 10, 20)):
    return {
        f"{m}@{k}": fn(ranked_lists, qrels, k)
        for fn, m in [(recall_at_k, "Recall"), (mrr_at_k, "MRR")]
        for k in ks
    }


def print_results(name, results):
    print(f"\n{'='*52}\n  {name}\n{'='*52}")
    for metric, val in results.items():
        bar = "█" * int(val * 30)
        print(f"  {metric:<12}  {val:.4f}  {bar}")

## 6 · Sparse Retrieval — BM25

**BM25** (Best Match 25) is the classic term-frequency ranking function.
It is fast, needs no GPU, and remains a tough baseline for legal text where
queries and documents share precise legal vocabulary.

In [ ]:
from rank_bm25 import BM25Okapi

def tokenise(text: str) -> list:
    return re.sub(r"[^\w\s]", " ", text.lower()).split()

print("Tokenising corpus …")
tokenised_corpus = [tokenise(t) for t in tqdm(corpus_texts)]

print("Building BM25 index …")
bm25 = BM25Okapi(tokenised_corpus)
print("Done.")

In [ ]:
def bm25_retrieve(query: str, top_k: int = 20) -> list:
    scores = bm25.get_scores(tokenise(query))
    return np.argsort(scores)[::-1][:top_k].tolist()


print("Running BM25 retrieval …")
bm25_ranked = [bm25_retrieve(q) for q in tqdm(queries)]

bm25_results = evaluate(bm25_ranked, qrel)
print_results("BM25 (sparse)", bm25_results)

In [ ]:
# ── Qualitative inspection ────────────────────────────────────────────────────
qi = 5   # ← try different indices

print(f"Query : {queries[qi]}\n")
for rank, pid in enumerate(bm25_ranked[qi][:5], 1):
    mark   = "✓ RELEVANT" if pid in qrel[qi] else "          "
    snippet = textwrap.shorten(corpus_texts[pid], width=140)
    print(f"  Rank {rank} {mark}  [{corpus_meta[pid][0][:35]}]")
    print(f"           {snippet}\n")

## 7 · Dense Retrieval — Sentence Transformers

Dense retrieval encodes queries and passages as **fixed-size vectors** and
retrieves by **cosine similarity**.

```
Query  ──[encoder]──► q_vec  (384-dim)
Passage──[encoder]──► p_vec  (384-dim)
score = cos(q_vec, p_vec)
```

We use **FAISS** for efficient nearest-neighbour lookup.

**Model 1 — general purpose:** `all-MiniLM-L6-v2`  (≈ 22 MB, very fast)

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

GENERAL_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading {GENERAL_MODEL} …")
st_model = SentenceTransformer(GENERAL_MODEL, device=device)
print("Embedding dim:", st_model.get_sentence_embedding_dimension())

In [ ]:
print("Encoding corpus passages …")
corpus_emb_general = st_model.encode(
    corpus_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # cosine sim via inner product
)
print(f"Shape: {corpus_emb_general.shape}")

In [ ]:
# Build FAISS flat index (exact search)
dim_gen = corpus_emb_general.shape[1]
index_general = faiss.IndexFlatIP(dim_gen)   # Inner Product = cosine after L2-norm
index_general.add(corpus_emb_general)
print(f"FAISS index: {index_general.ntotal} vectors, dim={dim_gen}")

In [ ]:
def dense_retrieve(query: str, index, model, top_k: int = 20) -> list:
    q_vec = model.encode([query], normalize_embeddings=True)
    _, I  = index.search(q_vec, top_k)
    return I[0].tolist()


print("Running dense retrieval (general model) …")
general_ranked = [dense_retrieve(q, index_general, st_model) for q in tqdm(queries)]

general_results = evaluate(general_ranked, qrel)
print_results("all-MiniLM-L6-v2 (general dense)", general_results)

## 8 · Legal-Domain Embeddings — Legal-BERT

`nlpaueb/legal-bert-base-uncased` was pre-trained on 12 GB of English legal text
(EU legislation, US court decisions, contracts).

We extract sentence embeddings via **mean pooling** over the final hidden states —
the same technique `sentence-transformers` uses internally.

| | MiniLM (general) | Legal-BERT |
|---|---|---|
| Pre-training data | general web | legal corpora |
| Size | ≈ 22 MB | ≈ 440 MB |
| Embedding dim | 384 | 768 |

In [ ]:
from transformers import AutoTokenizer, AutoModel

LEGAL_MODEL = "nlpaueb/legal-bert-base-uncased"
print(f"Loading {LEGAL_MODEL} …")
legal_tok = AutoTokenizer.from_pretrained(LEGAL_MODEL)
legal_enc = AutoModel.from_pretrained(LEGAL_MODEL).to(device).eval()
print("Done.")

In [ ]:
def mean_pool(model_output, attention_mask):
    """Mean-pool last hidden states, ignoring padding."""
    token_embs = model_output.last_hidden_state          # (B, L, H)
    mask = attention_mask.unsqueeze(-1).float()          # (B, L, 1)
    return (token_embs * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


@torch.no_grad()
def encode_legal(texts: list, batch_size: int = 32) -> np.ndarray:
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        enc  = legal_tok(texts[i:i+batch_size], padding=True,
                         truncation=True, max_length=512,
                         return_tensors="pt").to(device)
        out  = legal_enc(**enc)
        embs = mean_pool(out, enc["attention_mask"])
        embs = torch.nn.functional.normalize(embs, dim=-1)
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs)

In [ ]:
print("Encoding corpus with Legal-BERT (≈ 3–5 min on a 24 GB GPU) …")
corpus_emb_legal = encode_legal(corpus_texts, batch_size=64)
print(f"Shape: {corpus_emb_legal.shape}")

In [ ]:
dim_leg = corpus_emb_legal.shape[1]
index_legal = faiss.IndexFlatIP(dim_leg)
index_legal.add(corpus_emb_legal)
print(f"FAISS index: {index_legal.ntotal} vectors, dim={dim_leg}")

In [ ]:
@torch.no_grad()
def legal_retrieve(query: str, index, top_k: int = 20) -> list:
    enc   = legal_tok([query], padding=True, truncation=True,
                      max_length=512, return_tensors="pt").to(device)
    out   = legal_enc(**enc)
    q_vec = mean_pool(out, enc["attention_mask"])
    q_vec = torch.nn.functional.normalize(q_vec, dim=-1).cpu().numpy()
    _, I  = index.search(q_vec, top_k)
    return I[0].tolist()


print("Running dense retrieval (Legal-BERT) …")
legal_ranked = [legal_retrieve(q, index_legal) for q in tqdm(queries)]

legal_results = evaluate(legal_ranked, qrel)
print_results("Legal-BERT (legal dense)", legal_results)

## 9 · Head-to-Head Comparison

In [ ]:
import pandas as pd

systems = {
    "BM25":               bm25_results,
    "MiniLM (general)":   general_results,
    "Legal-BERT (legal)": legal_results,
}

rows = []
for name, res in systems.items():
    row = {"Model": name}
    row.update({k: f"{v:.3f}" for k, v in res.items()})
    rows.append(row)

df = pd.DataFrame(rows).set_index("Model")
print(df.to_string())

In [ ]:
import matplotlib.pyplot as plt

ks = [1, 5, 10, 20]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, prefix in zip(axes, ["Recall", "MRR"]):
    for name, res in systems.items():
        vals = [res[f"{prefix}@{k}"] for k in ks]
        ax.plot(ks, vals, marker="o", label=name)
    ax.set_title(f"{prefix}@k on CUAD subset")
    ax.set_xlabel("k")
    ax.set_ylabel(prefix)
    ax.set_xticks(ks)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("retrieval_comparison.png", dpi=120)
plt.show()

## 10 · Qualitative Error Analysis

Find a query where BM25 fails but Legal-BERT succeeds — this shows *when*
domain-specific embeddings help.

In [ ]:
def find_interesting(bm25_r, dense_r, qrels, k=10):
    """Query where BM25 fails @k but dense succeeds @k."""
    for i, (b, d, rel) in enumerate(zip(bm25_r, dense_r, qrels)):
        if (not any(p in rel for p in b[:k])) and any(p in rel for p in d[:k]):
            return i
    return 0

qi = find_interesting(bm25_ranked, legal_ranked, qrel)
print(f"Query  : {queries[qi]}\n{'─'*60}")

for label, ranked in [("BM25", bm25_ranked[qi]), ("Legal-BERT", legal_ranked[qi])]:
    print(f"\n  ── {label} top-5 ──")
    for rank, pid in enumerate(ranked[:5], 1):
        mark    = "✓ RELEVANT" if pid in qrel[qi] else "          "
        snippet = textwrap.shorten(corpus_texts[pid], width=120)
        print(f"    Rank {rank} {mark}  {snippet}")

## 11 · Mini-RAG Demo

Close the loop: retrieve the top passages with Legal-BERT, then feed them
to a small instruction-tuned LLM to generate a grounded answer.

> **GPU note:** `Mistral-7B-Instruct-v0.3` in 4-bit needs ≈ 5 GB.
> The retrieval indices are already loaded, so total VRAM ≈ 6–7 GB — well
> within 24 GB.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

GEN_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading {GEN_MODEL} in 4-bit …")
gen_tok   = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL, quantization_config=bnb_cfg, device_map="auto"
).eval()
print(f"Loaded.  VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
@torch.no_grad()
def rag_answer(question: str, top_k: int = 3, max_new: int = 256) -> str:
    # 1. Retrieve top-k passages
    pids = legal_retrieve(question, index_legal, top_k=top_k)
    context = "\n\n---\n\n".join(
        f"[Passage {i+1}]\n{corpus_texts[p]}" for i, p in enumerate(pids)
    )

    # 2. Build instruct-style prompt
    messages = [
        {
            "role": "system",
            "content": (
                "You are a legal contract analysis assistant. "
                "Answer the question using ONLY the provided contract passages. "
                "If the answer is not in the passages, say 'Not found in context'."
            ),
        },
        {
            "role": "user",
            "content": f"Contract passages:\n\n{context}\n\nQuestion: {question}",
        },
    ]

    # 3. Generate
    input_ids = gen_tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(device)
    out = gen_model.generate(
        input_ids, max_new_tokens=max_new, do_sample=False,
        temperature=None, top_p=None,
    )
    return gen_tok.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True)

In [ ]:
# ── Run on a dataset query ────────────────────────────────────────────────────
demo_q = queries[0]
print(f"Question:\n{demo_q}\n{'─'*60}")
print(f"\nRAG Answer:\n{rag_answer(demo_q)}")

In [ ]:
# ── Try your own question ─────────────────────────────────────────────────────
my_q = "What is the notice period required to terminate the agreement?"
print(f"Question:\n{my_q}\n{'─'*60}")
print(f"\nRAG Answer:\n{rag_answer(my_q)}")

## 12 · Key Takeaways

1. **BM25 is a strong baseline** — exact term matching works well when queries
   and documents share precise legal vocabulary.

2. **Dense retrieval generalises better** to paraphrase and semantic similarity —
   essential when queries and documents use different terminology.

3. **Domain pre-training matters** — Legal-BERT understands legal concepts
   ("indemnification", "force majeure") better than a general-purpose model.

4. **Chunking strategy is critical** — too short → loss of context;
   too long → signal dilution and BERT truncation errors.

5. **Retrieval quality is the ceiling for RAG** — if the right passage is not
   retrieved, the LLM cannot produce a correct answer.

---

### Further Reading
- CUAD paper — [Hendrycks et al., 2021](https://arxiv.org/abs/2103.06268)
- Legal-BERT — [Chalkidis et al., 2020](https://arxiv.org/abs/2010.02559)
- BEIR benchmark — [Thakur et al., 2021](https://arxiv.org/abs/2104.08663)